In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# FinanceGPT - Fine Tuning Notebook

This notebook is used for:
- Loading trainable language models
- Preparing LoRA fine-tuning
- Loading processed datasets
- Understanding AI training workflow
- Preparing FinanceGPT for custom learning

## Installing Required Libraries

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
!pip install transformers==4.40.2 -q

!pip install datasets -q

!pip install peft==0.11.1 -q

!pip install trl==0.8.6 -q

!pip install accelerate==0.30.1 bitsandbytes -q

## Importing Required Libraries

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from datasets import (
    load_dataset,
    load_from_disk
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

## Understanding Fine Tuning

Fine-tuning means adapting a pretrained AI model
to perform better on a custom dataset.

Instead of training from scratch,
we improve an existing model using finance-specific data.

## Loading Base Model

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

## Understanding LoRA

LoRA (Low Rank Adaptation) allows efficient fine-tuning
by training only small adapter layers instead of
updating the entire model.

## Configuring LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(
    model,
    lora_config
)

In [ ]:
model.print_trainable_parameters()

## Loading Processed Dataset

In [ ]:
dataset = load_dataset(
    "gbharti/finance-alpaca",
    split="train[:500]"
)

In [ ]:
print(dataset)

## Configuring Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./financegpt-output",
    per_device_train_batch_size=1,
    max_steps=20,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

## Preparing Trainer

In [ ]:
def format_prompt(example):
    return f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""

dataset = dataset.map(
    lambda x: {"text": format_prompt(x)}
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args
)

## Starting Fine Tuning

The model will now begin learning
from the finance-specific dataset.

In [ ]:
trainer.train()

## Saving Fine Tuned Model

In [ ]:
save_path = "/content/drive/MyDrive/financegpt-finetuned"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("FinanceGPT model saved to Google Drive successfully.")

## Final Observation

FinanceGPT has now completed:
- dataset preparation
- tokenization
- LoRA setup
- fine-tuning preparation
- custom finance training

This marks the beginning of FinanceGPT
becoming a domain-specific AI assistant.